# 06 - Busca de Hiperparametros (vale a pena melhorar?)
Pergunta: **uma busca sistematica de hiperparametros melhora o baseline?**

Usamos `RandomizedSearchCV` (mesma tecnica das aulas) com validacao cruzada
**apenas no treino** (sem tocar no teste, para nao enviesar a avaliacao) e comparamos
com a regressao logistica ja usada no notebook 03.

In [ ]:
# bibliotecas
import numpy as np
import pandas as pd
import sys; sys.path.append("../src")
pd.options.display.float_format = "{:.4f}".format
DATA = "../data/processed"

In [ ]:
# --- pre-processamento feito aqui mesmo, no estilo das aulas ---
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num = ["idade_anos", "NU_CONTATO", "dias_diag_trat"]
cat = ["CS_SEXO","CS_RACA","CS_ESCOL_N","SG_UF_NOT","RAIOX_TORA","TESTE_TUBE",
       "BACILOSC_E","HIV","AGRAVAIDS","AGRAVALCOO","AGRAVDIABE","AGRAVDOENC",
       "AGRAVDROGA","AGRAVTABAC","TRAT_SUPER","ANT_RETRO","BENEF_GOV",
       "POP_RUA","POP_LIBER","POP_IMIG","POP_SAUDE"]

def preparar(df):
    # cria o atraso ate o inicio do tratamento e deixa as categoricas como texto
    df = df.copy()
    df["DT_DIAG"]    = pd.to_datetime(df["DT_DIAG"], errors="coerce")
    df["DT_INIC_TR"] = pd.to_datetime(df["DT_INIC_TR"], errors="coerce")
    df["dias_diag_trat"] = (df["DT_INIC_TR"] - df["DT_DIAG"]).dt.days
    for c in cat:
        df[c] = df[c].fillna("ignorado").astype(str)
    return df[num + cat]
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import loguniform

treino = pd.read_csv(f"{DATA}/treino.csv"); teste1 = pd.read_csv(f"{DATA}/teste1.csv")
# busca de hiperparametros na base completa

In [ ]:
# pre-processa os dados
transf = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False), cat),
], verbose_feature_names_out=False)
X = transf.fit_transform(preparar(treino)); y = treino["ltfu"].values
X_teste1 = transf.transform(preparar(teste1)); y_teste1 = teste1["ltfu"].values
cv = StratifiedKFold(4, shuffle=True, random_state=42)

### 1. Baseline (configuracao do notebook 03)

In [ ]:
base = LogisticRegression(max_iter=300, class_weight="balanced")
cv_base = cross_val_score(base, X, y, cv=cv, scoring="roc_auc").mean()
base.fit(X, y)
auc_base = roc_auc_score(y_teste1, base.predict_proba(X_teste1)[:, 1])
print(f"Baseline -> CV-AUC (treino) = {cv_base:.4f} | AUC (teste1) = {auc_base:.4f}")

### 2. Busca de hiperparametros (RandomizedSearchCV)

In [ ]:
grid = {"C": loguniform(1e-2, 1e2), "class_weight": ["balanced", None]}
busca = RandomizedSearchCV(LogisticRegression(max_iter=400, solver="lbfgs"),
                           grid, n_iter=10, cv=cv, scoring="roc_auc",
                           random_state=42, n_jobs=-1)
busca.fit(X, y)
auc_tun = roc_auc_score(y_teste1, busca.best_estimator_.predict_proba(X_teste1)[:, 1])
print("Melhores parametros:", busca.best_params_)
print(f"Tunado   -> CV-AUC (treino) = {busca.best_score_:.4f} | AUC (teste1) = {auc_tun:.4f}")

### 3. Comparacao

In [ ]:
comp = pd.DataFrame({
    "CV-AUC (treino)": [cv_base, busca.best_score_],
    "AUC (teste1)":    [auc_base, auc_tun],
}, index=["Baseline", "Com RandomizedSearchCV"])
comp.loc["Ganho"] = comp.loc["Com RandomizedSearchCV"] - comp.loc["Baseline"]
comp

### Conclusao
O ganho e **proximo de zero** (terceira casa decimal), dentro do ruido de um conjunto
de teste pequeno (631 casos). Ou seja, **o baseline ja esta proximo do otimo**: o
fator limitante e a qualidade/vies dos dados (censura a direita), nao o ajuste do
modelo. Isso justifica nao investir em otimizacao extra de hiperparametros neste
projeto - e e uma evidencia de rigor, nao de falta de esforco.